In [1]:
import pathlib
import os
from typing import List

from datasets import load_dataset, load_from_disk
import evaluate
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

from hf_wrapper import GPTForSequenceClassification
from model import GPT, GPTConfig
from tokenizer import load_tokenizer, eod_token
from sklearn.metrics import precision_score, recall_score, f1_score


def load_pretrained_model(path: pathlib.Path, device: str = 'cuda') -> GPT:
    print(f"Loading pretrained model from {path}")
    checkpoint = torch.load(path, map_location=device)

    gptconf = GPTConfig(**checkpoint['model_args'])
    pretrained_model = GPT(gptconf)
    state_dict = checkpoint['model']

    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

    model_dict = pretrained_model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items()
                           if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(filtered_state_dict)
    pretrained_model.load_state_dict(model_dict)
    pretrained_model.to(device)

    return pretrained_model


# Define variables directly here instead of using argparse
args = {
    'vocab': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json"),  # Example for IPA
    'merges': pathlib.Path("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt"),  # Example for IPA
    'model': pathlib.Path("/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_50k/ckpt.pt"),  # IPA Model
    'task': "rte",  # Example task
    'epochs': 8,
    'eval_interval': 0.01,
    'context_size': 1024,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'hf_cache_dir': pathlib.Path('cache'),
    'dataset': 'iggy12345/glue-rte-ipa',  # Dataset path
    'from_disk': False,
    'no_subset': False,
    'device': 'cuda',
    'no_progress_bar': False
}

# ---- Models and Tokenizers ----
models_and_tokenizers = [               #/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k
    {"model_type": "ipa", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-ipa-number-preservation-merges.txt")},
    {"model_type": "normal", "model_path": "/fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt",
     "tokenizer_paths": ("/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-vocab.json",
                         "/fs/ess/PAS2836/ipa_gpt/tokenizers/bpe-normal-number-preservation-merges.txt")},
    {"model_type": "prebuilt", "model_path": "/fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt",
     "tokenizer_paths": None}
]

# Create a main directory for all model outputs
main_output_dir = pathlib.Path("./training_outputs_rte")
if not os.path.exists(main_output_dir):
    os.makedirs(main_output_dir)

# Loop through the models and tokenizers
for config in models_and_tokenizers:
    model_type = config["model_type"]
    model_path = config["model_path"]
    tokenizer_paths = config["tokenizer_paths"]

    # Dynamically set the output directory based on the model type
    output_dir = main_output_dir / f"output_{model_type}"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # ---- Load Tokenizer ----
    if model_type == "ipa" or model_type == "normal":
        vocab_path, merges_path = tokenizer_paths
        tokenizer = load_tokenizer(vocab_path, merges_path)
    elif model_type == "prebuilt":
        tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
        tokenizer.pad_token = tokenizer.eos_token

    # ---- Load model ----
    base_model = load_pretrained_model(model_path, args['device'])
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.config.padding_side = tokenizer.padding_side
    model = GPTForSequenceClassification(base_model).to(args['device'])

    # ---- Load dataset ----
    dataset = load_dataset(args['dataset'], cache_dir=str(args['hf_cache_dir']))

    def flatten_multi_features(examples, features: List[str]) -> List[str]:
        separator = f'\n\n{eod_token}\n\n'
        return [separator.join(example) for example in zip(*[examples[f] for f in features])]

    # ---- Preprocessing ----
    def preprocess_function(examples):
        if 'premise' in examples:
            feature = flatten_multi_features(examples, ['premise', 'hypothesis'])
        elif 'question' in examples:
            if 'sentence' in examples:
                feature = flatten_multi_features(examples, ['question', 'sentence'])
            else:
                feature = flatten_multi_features(examples, ['question', 'hypothesis'])
        elif 'sentence1' in examples:
            feature = flatten_multi_features(examples, ['sentence1', 'sentence2'])
        elif 'question1' in examples:
            feature = flatten_multi_features(examples, ['question1', 'question2'])
        else:
            feature = examples['sentence']

        return tokenizer(feature, truncation=True, max_length=args['context_size'])

    encoded_dataset = dataset.map(preprocess_function, batched=True)

    # ---- Data collator ----
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # ---- Metrics ----
    metric = evaluate.load("glue", args['task'])

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = torch.from_numpy(logits).argmax(dim=-1)
        # return metric.compute(predictions=predictions, references=labels)
        hf_metrics = metric.compute(predictions=predictions, references=labels)
        hf_metrics["precision"] = precision_score(labels, predictions)
        hf_metrics["recall"] = recall_score(labels, predictions)
        hf_metrics["f1"] = f1_score(labels, predictions)
        return hf_metrics

    # ---- Training arguments ----
    training_args = TrainingArguments(
        output_dir=str(output_dir),  # Dynamically set output directory
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",  # Save every 'save_steps'
        save_steps=100,  # Save every 100 steps
        save_total_limit=1,  # Keep only 1 checkpoint
        metric_for_best_model="accuracy",
        load_best_model_at_end=True,  # Automatically load the best model after training
        learning_rate=args['learning_rate'],
        per_device_train_batch_size=args['batch_size'],
        per_device_eval_batch_size=args['batch_size'],
        num_train_epochs=args['epochs'],
        weight_decay=0.01,
        logging_steps=100, 
        logging_dir='./logs', 
        disable_tqdm=False,  
        warmup_ratio=0.3,  
        save_safetensors=False  
    )

    # ---- Trainer ----
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # ---- Train ----
    trainer.train(resume_from_checkpoint=False)  
    results = trainer.evaluate(encoded_dataset["validation"])
    print(f"Evaluation results for {model_type}: {results}")
    


/users/PAS2836/krishnakb/ondemand/krishna_proj/cleanenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_ipa_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


Map: 100%|██████████| 277/277 [00:00<00:00, 5388.29 examples/s]
/tmp/slurmtmp.1402334/ipykernel_879635/4235461078.py:172: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: orugantikoundinya7 (orugantikoundinya7-ohio-state-buckeyes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,0.845400,0.727917,0.555957,0.526316,0.610687,0.565371
200,0.779600,0.684745,0.563177,0.531646,0.641221,0.581315
300,0.694000,0.711846,0.577617,0.555556,0.534351,0.544747
400,0.665500,0.724179,0.620939,0.616071,0.526718,0.567901
500,0.593500,0.837979,0.537906,0.506726,0.862595,0.638418
600,0.640200,0.812448,0.509025,0.488372,0.801527,0.606936
700,0.389200,1.017997,0.642599,0.586022,0.832061,0.687697
800,0.432000,0.851124,0.613718,0.573171,0.717557,0.637288
900,0.443300,0.926716,0.646209,0.622222,0.641221,0.631579
1000,0.283100,1.670553,0.628159,0.604478,0.618321,0.611321


Evaluation results for ipa: {'eval_loss': 2.7345216274261475, 'eval_accuracy': 0.6895306859205776, 'eval_precision': 0.6470588235294118, 'eval_recall': 0.7557251908396947, 'eval_f1': 0.6971830985915493, 'eval_runtime': 4.5356, 'eval_samples_per_second': 61.072, 'eval_steps_per_second': 7.717, 'epoch': 8.0}
Loading pretrained model from /fs/scratch/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_medium_50k/ckpt.pt
number of parameters: 353.24M


Map: 100%|██████████| 277/277 [00:00<00:00, 7719.44 examples/s]
/tmp/slurmtmp.1402334/ipykernel_879635/4235461078.py:172: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
100,0.779100,0.765725,0.530686,0.502674,0.717557,0.591195
200,0.750000,0.679595,0.606498,0.574324,0.648855,0.609319
300,0.683800,0.693124,0.570397,0.540000,0.618321,0.576512
400,0.602300,0.768397,0.574007,0.540373,0.664122,0.595890
500,0.598800,0.829350,0.563177,0.527174,0.740458,0.615873
600,0.616700,0.851350,0.552347,0.515556,0.885496,0.651685
700,0.395600,1.012023,0.657040,0.632353,0.656489,0.644195
800,0.350800,1.120746,0.646209,0.617021,0.664122,0.639706
900,0.434000,0.968777,0.638989,0.597484,0.725191,0.655172
1000,0.198300,2.132606,0.657040,0.673077,0.534351,0.595745


Evaluation results for normal: {'eval_loss': 2.3126027584075928, 'eval_accuracy': 0.703971119133574, 'eval_precision': 0.6814814814814815, 'eval_recall': 0.7022900763358778, 'eval_f1': 0.6917293233082706, 'eval_runtime': 3.744, 'eval_samples_per_second': 73.985, 'eval_steps_per_second': 9.348, 'epoch': 8.0}
Loading pretrained model from /fs/ess/PAS2836/ipa_gpt/checkpoints/openwebtext_normal_multi_node_12_5_prebuilt_medium/ckpt.pt


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory